# P12-P13 · ICF y distribución de la frecuencia programada

> **¿Qué parte del ICF se puede calcular con GTFS y cómo se distribuye la frecuencia programada?**

frequencies.txt, trips.txt y routes.txt del GTFS vigente, más la definición oficial del ICF.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/frecuencia_programada/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# 1. Detectar el entorno y obtener una copia limpia del repositorio.
# Esta celda también funciona localmente si se abre el notebook dentro del repo.
import os
import platform
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
if not Path('pyproject.toml').exists():
    destino_repo = Path('lab-dtpm')
    if not destino_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git', str(destino_repo)], check=True)
    os.chdir(destino_repo)

ROOT = Path.cwd().resolve()
PAQUETES = ['pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'XlsxWriter', 'matplotlib', 'polars>=1.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES], check=True)
print('Entorno:', 'Google Colab' if EN_COLAB else 'Python local/Jupyter')
print('Python:', platform.python_version())
print('Repositorio:', ROOT)


In [ ]:
# 2. Importar y registrar versiones: evita resultados imposibles de auditar.
import matplotlib
import numpy as np
import pandas as pd
import polars as pl

print({'pandas': pd.__version__, 'polars': pl.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})


## Alternativas para integrar los datos

La vía reproducible es clonar GitHub y ejecutar `scripts/fetch_data.py`, como hace la siguiente celda. Si la red institucional lo impide, consulte la [guía portable](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/docs/ejecucion-portable.md): explica carga desde el equipo, montaje de Google Drive, descarga directa y validación SHA-256. No cambie la fuente por datos simulados sin declararlo.


In [ ]:
# 3. Obtener las fuentes originales declaradas por el caso.
# Si la fuente externa falla, el error se conserva: no se sustituyen datos
# reales por sintéticos. Los CSV versionados permiten ejecutar el laboratorio
# comparativo, pero no reemplazan la adquisición del caso completo.
fuentes = ['gtfs-red-movilidad']
for fuente in fuentes:
    try:
        subprocess.run([sys.executable, 'scripts/fetch_data.py', fuente], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'No se pudo obtener {fuente}. Revise la URL, la red y las alternativas de carga descritas arriba.'
        ) from exc


## 1 · El problema

El ICF oficial parece una razón simple, pero su numerador y denominador son mediciones
GPS. El GTFS solo contiene el programa. Confundir ambos convierte una promesa de oferta
en una medición de cumplimiento inexistente. El mismo archivo sí permite contestar P13:
cómo se distribuyen los intervalos programados y qué servicios tienen mayor espera.


## 2 · La solución que se va a aplicar

Se validan las ventanas, se calculan sus salidas con fin exclusivo y se construye una
matriz de alcance del ICF. La distribución se resume con percentiles y tres unidades de
análisis. La punta mañana se filtra por intersección temporal, no por hora de inicio.


## Implementaciones comparables

La ruta principal conserva el caso completo en Pandas. El laboratorio comparable ejecuta Pandas y Polars en este runtime sobre el mismo CSV. R usa el mismo contrato y se valida en CI; el runtime Python estándar de Colab no ejecuta R de forma nativa.

#### Contrato común del ejercicio

- **Objetivo.** Contrastar estadísticos de la distribución de intervalos programados.
- **Fuente que leen las tres implementaciones.**
  [`resultados/distribucion.csv`](resultados/distribucion.csv), CSV UTF-8 versionado y producido
  por la resolución completa del caso. Los datos originales, su publicador, fecha,
  licencia, URL y huella se documentan en [`case.json`](case.json) y, cuando existe,
  `procedencia.json`. Este CSV es un **resultado reproducible**, no se presenta como
  una nueva fuente oficial.
- **Alcance.** percentiles del intervalo programado. Los tres scripts comparables no sustituyen la adquisición,
  limpieza ni controles completos de los pasos Pandas: aíslan el indicador para que
  una diferencia observada sea atribuible al motor.
- **Fórmula o regla.** El percentil p con interpolación lineal se ubica en h=(n−1)p; se interpola entre floor(h) y ceil(h). La convención debe declararse.
- **Interpretación.** La mediana y el P90 muestran asimetría que el promedio puede ocultar.
- **Error frecuente.** Comparar percentiles calculados con métodos de interpolación distintos puede producir discrepancias legítimas.

| Campo | Tipo observado | Función en el ejercicio |
| --- | --- | --- |
| `estadistico` | `str` | categoría/eje X |
| `intervalo_min` | `float64` | medida/eje Y |
| `unidad` | `str` | campo auxiliar del contrato |

#### Qué cambia entre motores

| Dimensión | Pandas | Polars | R (`dplyr`/`readr`) |
| --- | --- | --- | --- |
| Modelo | DataFrame mutable por operaciones | expresiones columnares y plan optimizable | tubería declarativa sobre `tibble` |
| Nulos | `NaN`/`NA`; `groupby` exige decidir `dropna` | `null` separado de `NaN` | `NA`; muchas funciones exigen `na.rm` explícito |
| Orden | varias operaciones lo conservan, pero se ordena explícitamente | no se presupone tras agrupar | se fija con `arrange` |
| Gráfico | Matplotlib | Matplotlib tras extraer sólo X/Y | ggplot2 y su gramática de capas |
| Ventaja típica | ecosistema amplio y aprendizaje gradual | paralelismo, tipos y consultas grandes | estadística, comunicación y análisis reproducible |
| Límite aquí | el índice puede introducir alineaciones implícitas | añade una dependencia y otra semántica de tipos | requiere runtime y paquetes R independientes |

La equivalencia se valida sobre CSV normalizados con tolerancia numérica `1e-9`; no
se exige que la sintaxis ni los píxeles sean idénticos. Para ejecutar en un entorno
limpio consulte [`docs/ejecucion-portable.md`](../../docs/ejecucion-portable.md).

### Pandas + Matplotlib

```python
"""Laboratorio comparable Pandas + Matplotlib del caso frecuencia_programada.

Lee el mismo contrato CSV que las versiones Polars y R. La resolución completa del
caso permanece en ``solution.py`` y en los pasos del cuaderno; este archivo aísla el
indicador central para que la comparación entre motores sea controlada.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Buscar la raíz desde el propio archivo evita rutas C:\... específicas del autor y
# funciona igual después de clonar el repositorio en Linux, Windows o Google Colab.
ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "frecuencia_programada" / "resultados" / "distribucion.csv"
REQUIRED = {"estadistico", "intervalo_min"}


def analizar() -> pd.DataFrame:
    """Carga, valida y transforma el indicador; no modifica el archivo de entrada."""
    datos = pd.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # La expresión conserva explícitos filtro, agrupación, orden y selección para que
    # el estudiante pueda alinearlos con las expresiones de Polars y dplyr.
    resultado = datos.assign(_orden=datos["estadistico"].map({"mínimo": 0, "P10": 1, "P25": 2, "P50": 3, "media": 4, "P75": 5, "P90": 6, "P95": 7, "máximo": 8})).sort_values("_orden")[["estadistico", "intervalo_min"]]
    return resultado.reset_index(drop=True)


def graficar(resultado: pd.DataFrame, destino: Path) -> Path:
    """Representa exactamente las columnas comparadas y explicita sus unidades."""
    x = resultado["estadistico"].tolist()
    y = resultado["intervalo_min"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([str(value) for value in x], y, color="#11253B")
    ax.set_xlabel("Estadístico")
    ax.set_ylabel("Intervalo entre salidas (minutos)")
    ax.set_ylim(bottom=0)
    ax.set_title("La distribución revela más que un promedio")
    ax.grid(axis="y", alpha=0.25)
    fig.text(0.01, 0.01, "Fuente: Resultados reproducibles del caso P12/P13 · Período: publicación GTFS analizada", fontsize=8, color="#58677A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado.to_string(index=False))
    if args.salida:
        resultado.to_csv(args.salida, index=False)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `python cases/frecuencia_programada/cuaderno/solucion_pandas.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/frecuencia_programada/cuaderno/solucion_pandas.py', '--grafico', 'solucion_pandas-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_pandas-colab.png'))


### Polars + Matplotlib

```python
"""Laboratorio comparable Polars + Matplotlib del caso frecuencia_programada.

Usa el mismo CSV, columnas, reglas y escala que Pandas y R. Polars expresa las
transformaciones como un plan de expresiones columnares; esa diferencia de método es
el objeto de aprendizaje, no una razón para cambiar la pregunta.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "frecuencia_programada" / "resultados" / "distribucion.csv"
REQUIRED = {"estadistico", "intervalo_min"}


def analizar() -> pl.DataFrame:
    """Carga y valida antes de transformar para producir un error explicativo."""
    datos = pl.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # select al final fija el contrato de salida y evita comparar columnas auxiliares.
    return datos.with_columns(pl.col("estadistico").replace_strict({"mínimo": 0, "P10": 1, "P25": 2, "P50": 3, "media": 4, "P75": 5, "P90": 6, "P95": 7, "máximo": 8}).alias("_orden")).sort("_orden").select(["estadistico", "intervalo_min"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    """Convierte sólo las series del gráfico; el análisis permanece en Polars."""
    x = resultado["estadistico"].to_list()
    y = resultado["intervalo_min"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([str(value) for value in x], y, color="#11253B")
    ax.set_xlabel("Estadístico")
    ax.set_ylabel("Intervalo entre salidas (minutos)")
    ax.set_ylim(bottom=0)
    ax.set_title("La distribución revela más que un promedio")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P12/P13 · Período: publicación GTFS analizada",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.salida:
        resultado.write_csv(args.salida)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/frecuencia_programada/cuaderno/solucion_polars.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/frecuencia_programada/cuaderno/solucion_polars.py', '--grafico', 'solucion_polars-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_polars-colab.png'))


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Laboratorio comparable R + dplyr/readr + ggplot2 del caso frecuencia_programada.
#
# La resolución completa sigue en solution.py y en los pasos Pandas. Este script
# cambia el motor sobre el mismo contrato CSV para estudiar sintaxis y semántica sin
# cambiar datos, indicador ni escala gráfica.

required <- c("readr", "dplyr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  stop(
    "Faltan paquetes: ", paste(missing, collapse = ", "),
    ". Instálelos con install.packages(c('readr', 'dplyr', 'ggplot2'))."
  )
}

# Buscar pyproject.toml hacia arriba hace portable el script entre terminal, RStudio,
# Posit Cloud, IRkernel y un clon en Colab. LAB_ROOT permite una ruta explícita sin
# codificar la unidad o el directorio del desarrollador.
find_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "pyproject.toml"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("No se encontró la raíz del repositorio.")
    current <- parent
  }
}
root <- Sys.getenv("LAB_ROOT", unset = find_root())
source_csv <- file.path(root, "cases", "frecuencia_programada", "resultados", "distribucion.csv")

# readr conserva nombres y tipos previsibles. La validación temprana produce un error
# que explica el contrato roto, en vez de fallar más tarde dentro de una gráfica.
datos <- readr::read_csv(source_csv, show_col_types = FALSE, progress = FALSE)
required_columns <- c("estadistico", "intervalo_min")
missing_columns <- setdiff(required_columns, names(datos))
if (length(missing_columns)) stop("Faltan columnas: ", paste(missing_columns, collapse = ", "))

# dplyr expresa la misma selección, filtro, agrupación y orden que las otras vías.
resultado <- datos |> dplyr::mutate(.orden = match(.data$estadistico, c("mínimo", "P10", "P25", "P50", "media", "P75", "P90", "P95", "máximo"))) |> dplyr::arrange(.data$.orden) |> dplyr::select(.data$estadistico, .data$intervalo_min)
print(resultado)

args <- commandArgs(trailingOnly = TRUE)
value_after <- function(flag) {
  pos <- match(flag, args)
  if (is.na(pos)) return(NULL)
  if (pos == length(args)) stop(flag, " necesita una ruta de salida")
  args[pos + 1]
}

csv_output <- value_after("--salida")
if (!is.null(csv_output)) readr::write_csv(resultado, csv_output)

plot_output <- value_after("--grafico")
if (!is.null(plot_output)) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["estadistico"]], y = .data[["intervalo_min"]])
  ) +
    ggplot2::geom_col(fill = "#11253B") +
    ggplot2::labs(
      title = "La distribución revela más que un promedio",
      x = "Estadístico", y = "Intervalo entre salidas (minutos)",
      caption = "Fuente: Resultados reproducibles del caso P12/P13 · Período: publicación GTFS analizada"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(plot_output, grafico, width = 9, height = 5, dpi = 150)
  cat(plot_output, "\n")
}
```

Ejecutar con `Rscript cases/frecuencia_programada/cuaderno/solucion.R`.


### Ejecutar R

Use `solucion.R` desde R/RStudio, Posit Cloud o Jupyter con IRkernel. También se incluye `solucion.Rmd`. En Colab, seleccione un runtime R compatible o use la instalación experimental documentada; se marca como compatibilidad documentada, no como ejecución validada.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.frecuencia_programada import solution as frecuencia

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Leer solo tres tablas


El ICF programado no necesita el archivo de 1,1 millones de horarios.

> ⚠️ Cargar por costumbre la tabla grande duplica memoria sin aportar una columna al cálculo.


In [ ]:
tablas = frecuencia.cargar()
{k: v.shape for k, v in tablas.items()}


### Paso 2 · Validar las ventanas


Inicio, fin e intervalo se tipan antes de resumir.

> ⚠️ Una ventana inválida no se descarta sin contarla.


In [ ]:
ventanas = frecuencia.ventanas_programadas(tablas)
ventanas[['valida','salidas_programadas']].agg(['count','sum'])


### Paso 3 · Separar programa de observación


La matriz impide presentar el programa como ICF.

> ⚠️ 142 366 salidas programadas no son 142 366 mediciones GPS.


In [ ]:
resultados = frecuencia.analizar()
resultados['alcance_icf']


### Paso 4 · Conservar el ICF como no calculable


La ausencia de GPS es un resultado de alcance, no un error de ejecución.

> ⚠️ Rellenar con cero produciría un ICF de 0 % que los datos jamás observaron.


In [ ]:
resultados['alcance_icf'].query("calculable_con_gtfs == 'NO'")


### Paso 5 · Describir la distribución


Percentiles muestran la asimetría entre 3,10 y 330 minutos.

**En Excel:** `=PERCENTIL.INC(rango_intervalo_min;0,5)`

Debe dar: **15 minutos**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ La media de 21,17 minutos no representa la ventana típica de 15.


In [ ]:
resultados['distribucion']


### Paso 6 · Cambiar la unidad de análisis


Ventana, servicio y salida responden preguntas distintas.

> ⚠️ Promediar servicios sin declarar peso da el mismo voto a uno diario y a uno intensivo.


In [ ]:
resultados['ponderaciones']


### Paso 7 · Filtrar la punta por intersección


Entra toda ventana que toca 08:00–09:00.

> ⚠️ Filtrar solo `start_time` pierde ventanas que comenzaron antes de las 08:00.


In [ ]:
resultados['punta_manana'].head(15)


### Paso 8 · Publicar una conclusión proporcional


Se distingue frecuencia larga de incumplimiento.

> ⚠️ Un intervalo programado largo puede ser diseño de servicio; no demuestra falla operacional.


In [ ]:
resultados['ventanas_invalidas']


## 4 · El resultado

El GTFS permite construir 142 366 salidas programadas y describir sus intervalos. No contiene mediciones GPS: el ICF oficial permanece no calculable.

## 5 · Qué queda aprendido

1. Programa no es operación observada.
2. La unidad de análisis cambia el intervalo típico.
3. Las ventanas se filtran por intersección temporal.
4. Un vacío metodológico no se rellena con cero.
